# Robotis FFW: Gym Wrapper Demo

This notebook demonstrates how to instantiate the `Robotis FFW` task environment just like a Gym environment, without any dependency on the full RSL-RL training loop.

We will use `ManagerBasedRlEnv` directly and wrap it with `MjlabGymVecEnv`.

In [1]:
import torch
from mjlab.envs import ManagerBasedRlEnv
from mjlab.utils.gym_wrapper import MjlabGymVecEnv
from mjlab.tasks.ffw.ffw_env_cfg import FFW_MINIMAL_ENV_CFG

# Set up device
device = "cuda:0" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cuda:0


## 1. Configure and Instantiate the Environment

We'll use a smaller number of environments for this demo (e.g., 32 instead of 4096) to make it run faster.

In [2]:
# Use fewer environments for the demo
env_cfg = FFW_MINIMAL_ENV_CFG
env_cfg.scene.num_envs = 32  # Override number of envs

# Create the environment
env = ManagerBasedRlEnv(cfg=env_cfg, device=device)

# Wrap it in the Gym compatibility layer
# This makes it behave like a gymnasium.Env with tensor outputs
env = MjlabGymVecEnv(env)

print(f"Environment created with {env.num_envs} parallel environments.")
print(f"Action Space: {env.action_space}")
print(f"Observation Space: {env.observation_space}")

/puffertank/mjlab/src/mjlab/scene/scene.py:48: UserWarning: Entity 'robot' has 2 keyframes; only the first one will be used.
  self._add_entities()


Warp 1.12.0.dev20260114 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA GeForce RTX 5060 Ti" (15 GiB, sm_120, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.12.0.dev20260114
Warp DeprecationWarning: The namespace `warp.context` will soon be removed from the public API. It can still be accessed from `warp._src.context` but might be changed or removed without notice.
Warp DeprecationWarning: The symbol `warp.context.runtime` will soon be removed from the public API. It can still be accessed from `warp._src.context.runtime` but might be changed or removed without notice.
Module mujoco_warp._src.smooth 9d8c26b load on device 'cuda:0' took 3.67 ms  (cached)
Module mujoco_warp._src.collision_driver de6dc98 load on device 'cuda:0' took 0.28 ms  (cached)
Module _nxn_broadphase__locals__kernel_db231363 db23136 load on device 'cuda:0' took 0.27 ms  (cached)
Module ccd_kernel_builder__locals__ccd_kernel_e9ec7bc5 e9ec7bc l

## 2. Interaction Loop

Now we can use the familiar `env.reset()` and `env.step()` loop. The key difference is that inputs and outputs are **PyTorch tensors** on the GPU, not NumPy arrays.

In [ ]:
# 1. Reset the environment
obs, info = env.reset()

print("Initial Observation Groups:", obs.keys())
if "actor" in obs and isinstance(obs["actor"], dict):
    print("\nActor Observation Terms:")
    for k, v in obs["actor"].items():
        print(f"  - {k}: {v.shape}")
elif "actor" in obs:
    print("Actor Obs Shape: ", obs["actor"].shape)


# 2. Run a loop with random actions
num_steps = 100
total_reward = torch.zeros(env.num_envs, device=device)

print(f"\nRunning {num_steps} simulation steps...")
for i in range(num_steps):
    # Sample random actions in [-1, 1] range
    # Shape: (num_envs, action_dim)
    actions = 2 * torch.rand(env.num_envs, env.single_action_space.shape[0], device=device) - 1
    
    # Step the environment
    # Returns: obs (dict), reward (tensor), terminated (bool tensor), truncated (bool tensor), info (dict)
    obs, reward, terminated, truncated, info = env.step(actions)
    
    total_reward += reward
    
    # Check for any reset environments
    if terminated.any() or truncated.any():
        reset_ids = (terminated | truncated).nonzero(as_tuple=False).squeeze(-1)
        if len(reset_ids) > 0 and i % 50 == 0:
             print(f"Step {i}: {len(reset_ids)} environments reset. Mean Ep Reward: {total_reward[reset_ids].mean().item():.2f}")
             total_reward[reset_ids] = 0.0

print(f"\nCompleted {num_steps} steps.")
print(f"Final Mean Reward: {total_reward.mean().item():.2f}")

if "reward_terms" in info:
    print("\nReward Breakdown (last step first env):")
    for k, v in info["reward_terms"].items():
        print(f"  - {k}: {v[0].item():.4f}")

Initial Observation Keys: dict_keys(['actor', 'critic'])
Actor Obs Shape:  torch.Size([32, 63])
Critic Obs Shape: torch.Size([32, 63])

Running 100 simulation steps...
Step 0: 11 environments reset. Mean Ep Reward: -0.41

Completed 100 steps.
Final Mean Reward: -3.04

Completed 100 steps.
Final Mean Reward: -3.04


## 3. Visualize a Single Environment

We can render a single frame to an RGB array to check if the simulation is visualized correctly.

In [4]:
import matplotlib.pyplot as plt

# Render one frame (requires render_mode="rgb_array")
# The environment was created with default ViewerConfig
# which enables rendering if render_mode is set.

# Note: Offscreen rendering may require EGL setup
# If it fails, check if MUJOCO_GL="egl" is set in env variables

try:
    rgb = env.render()
    if rgb is not None:
        plt.imshow(rgb)
        plt.axis('off')
        plt.show()
    else:
        print("Render returned None (check if render_mode='rgb_array' was passed)")
except Exception as e:
    print(f"Rendering failed: {e}")

Render returned None (check if render_mode='rgb_array' was passed)


## 4. Close and Cleanup
Remember to close the environment to release resources.

In [5]:
env.close()
print("Environment closed.")

Environment closed.
